Let's look into what is in our dataset and what useful information we could extract

In [ ]:
import numpy as np
import pandas as pd

# what does human legit data look like?
# df = pd.read_csv("../dataset/human-generated/legit.csv")
# print(df.head())
# print(df.info())
# print(df['label'].value_counts())

# 7 labels: sender, receiver, date, subject, body, urls, and labels

# what does human phishing look like?
# df = pd.read_csv("../dataset/human-generated/phishing.csv")
# print(df.head())
# print(df.info())
# print(df['label'].value_counts())

# 7 labels: sender, receiver, date, subject, body, urls, and labels

# what does llm-generated legit look like?
# df = pd.read_csv("../dataset/llm-generated/legit.csv")
# print(df.head())
# print(df.info())
# print(df['label'].value_counts())

# 2 labels: text, label

# what does llm-generated phishing look like?
# df = pd.read_csv("../dataset/llm-generated/phishing.csv")
# print(df.head())
# print(df.info())
# print(df['label'].value_counts())

# 2 labels: text, label



There is mismatch between human-generated and llm-generated data (7 columns vs. 2 columns), so let's do some cleaning

In [2]:
import pandas as pd
import re

human_legit = pd.read_csv("../dataset/human-generated/legit.csv")
human_phishing = pd.read_csv("../dataset/human-generated/phishing.csv")
llm_legit = pd.read_csv("../dataset/llm-generated/legit.csv")
llm_phishing = pd.read_csv("../dataset/llm-generated/phishing.csv", engine="python", on_bad_lines="skip")

chatgpt = pd.read_csv("../dataset/llm-generated/chatgpt_generated.csv")
claude = pd.read_csv("../dataset/llm-generated/claude_generated.csv")
copilot = pd.read_csv("../dataset/llm-generated/copilot_generated.csv")
grok = pd.read_csv("../dataset/llm-generated/grok_generated.csv")

# We want just three labels across each dataset: clean_text | label | source 
def build_text(df, text_col=None):
    """
    Converts any messy email dataframe into a single text field safely.
    Handles:
    - subject/body/urls format
    - single-column text format
    - broken rows
    """

    # Case 1: structured email dataset
    if {"subject", "body"}.issubset(df.columns):
        subject = df["subject"].fillna("").astype(str)
        body = df["body"].fillna("").astype(str)
        urls = df["urls"].fillna("").astype(str) if "urls" in df.columns else ""

        return subject + " " + body + " " + urls

    # Case 2: LLM-style dataset or single text column
    elif text_col and text_col in df.columns:
        return df[text_col].fillna("").astype(str)

    # Case 3: totally broken rows (fallback)
    else:
        return df.astype(str).apply(lambda row: " ".join(row.values), axis=1)
    
human_legit["clean_text"] = build_text(human_legit)
human_phishing["clean_text"] = build_text(human_phishing)

llm_legit = llm_legit.rename(columns={"text": "text"})
llm_phishing = llm_phishing.rename(columns={"text": "text"})
llm_legit["clean_text"] = build_text(llm_legit, text_col="text")
llm_phishing["clean_text"] = build_text(llm_phishing, text_col="text")

human_legit["label"] = "legit"
human_legit["source"] = "human"
human_phishing["label"] = "phishing"
human_phishing["source"] = "human"

llm_legit["label"] = "legit"
llm_legit["source"] = "llm"
llm_phishing["label"] = "phishing"
llm_phishing["source"] = "llm"

chatgpt["source"] = "llm"
chatgpt["ai_agent"] = "chatgpt"

claude["source"] = "llm"
claude["ai_agent"] = "claude"

copilot["source"] = "llm"
copilot["ai_agent"] = "copilot"

grok["source"] = "llm"
grok["ai_agent"] = "grok"

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", " URL ", text)
    text = re.sub(r"\S+@\S+", " EMAIL ", text)
    text = re.sub(r"\d+", " NUMBER ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

for df in [human_legit, human_phishing, llm_legit, llm_phishing]:
    df["clean_text"] = df["clean_text"].apply(clean_text)

required_cols = ["clean_text", "label", "source"]

# Drop rows missing required columns
df = df.dropna(subset=required_cols)

df["clean_text"] = df["clean_text"].astype(str)
df = df[df["clean_text"].str.strip() != ""]

# remove non-email text
df["word_count"] = df["clean_text"].str.split().str.len()

df = df[df["word_count"] >= 15]

df = pd.concat([
    human_legit,
    human_phishing,
    llm_legit,
    llm_phishing,
    chatgpt,
    copilot,
    claude,
    grok
], ignore_index=True)

label_map = {
    "legit": 0,
    "phishing": 1
}
df["label_id"] = df["label"].map(label_map)

# clean filtering
df = df[df["clean_text"].str.len() > 10]
df = df[df["clean_text"].str.split().str.len() > 5]

# let's do some checking
print(df.shape)
print(df["label"].value_counts())
print(df.groupby(["source", "label"]).size())

df.to_csv("../dataset/processed_dataset.csv", index=False)

(5989, 12)
label
phishing    3989
legit       2000
Name: count, dtype: int64
source  label   
human   legit       1000
        phishing    1000
llm     legit       1000
        phishing    2989
dtype: int64
